# US-004 - EDA Base 1

Analise exploratoria da base historica completa, com foco em qualidade dos dados, distribuicao de `perfil_latente` e relacao entre features comportamentais de clustering e os perfis latentes.

In [ ]:
import matplotlib
matplotlib.use('Agg')

import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

os.makedirs('reports', exist_ok=True)
sns.set_theme(style='whitegrid')

## Carregamento e visao geral

In [ ]:
base1_path = 'data/raw/ford_clientes_historico_completo.csv'
df = pd.read_csv(base1_path)

print(f'Linhas: {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]:,}')
df.head()

In [ ]:
df.dtypes.to_frame('tipo')

## Missing values

Tabela de valores ausentes por coluna, ordenada pelo percentual de missing.

In [ ]:
missing_values = (
    df.isna()
    .sum()
    .to_frame('missing_qtd')
    .assign(missing_pct=lambda x: (x['missing_qtd'] / len(df) * 100).round(2))
    .sort_values(['missing_pct', 'missing_qtd'], ascending=False)
)

missing_values

## Distribuicao do perfil_latente

In [ ]:
perfil_counts = (
    df['perfil_latente']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename_axis('perfil_latente')
    .reset_index(name='percentual')
)

plt.figure(figsize=(8, 4.5))
ax = sns.barplot(data=perfil_counts, x='perfil_latente', y='percentual', color='#003478')
ax.set_title('Distribuicao do perfil_latente - Base 1')
ax.set_xlabel('Perfil latente')
ax.set_ylabel('Percentual de clientes')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f%%', padding=3)
plt.tight_layout()
plt.savefig('reports/base1_distribuicao_perfil_latente.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

perfil_counts

## Features de clustering x perfil_latente

O heatmap abaixo compara as features comportamentais usadas para segmentacao com `perfil_latente` codificado ordinalmente apenas para calculo exploratorio de correlacao.

In [ ]:
clustering_features = [
    'meses_ate_primeira_revisao',
    'perdeu_primeira_revisao',
    'voltou_tarde_revoltado',
    'trouxe_oleo_externo',
    'pede_desconto_revisao',
    'sensibilidade_desconto_pos',
    'qtde_revisoes_24m',
    'share_revisoes_rede_24m',
    'gasto_manutencao_rede_24m',
    'satisfacao_marca_24m',
]

perfil_map = {'fiel': 0, 'economico': 1, 'abandono': 2, 'esquecido': 3}
corr_df = df[clustering_features].copy()
corr_df['perfil_latente'] = df['perfil_latente'].map(perfil_map)
corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.5)
plt.title('Correlacao entre features de clustering e perfil_latente')
plt.tight_layout()
plt.savefig('reports/base1_heatmap_clustering_perfil_latente.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

corr[['perfil_latente']].sort_values('perfil_latente', key=lambda s: s.abs(), ascending=False)